# 10 · 数据管线：从多传感器表到一帧可审计 Bundle

模型效果的很多“玄学问题”其实来自数据接口：sample token 错了、时间戳没对齐、calibrated sensor 版本混用、某个传感器静默缺失。这个 notebook 用 pandas 构造一个简化的 nuScenes-like 表结构，练习把一帧场景组装成可审计的 sensor bundle。

学习目标：

- 理解 scene、sample、sample_data、ego_pose、calibrated_sensor 的关系。
- 按 frame、sensor 和 timestamp 组装输入，而不是依赖文件名猜测。
- 检测 missing frame、时间偏移、重复记录和校准引用缺失。
- 为后续真实 nuScenes devkit 适配器定义清晰的 I/O。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

plt.rcParams['figure.figsize'] = (10, 4.5)

SENSORS = ['CAM_FRONT', 'LIDAR_TOP', 'RADAR_FRONT']
FRAME_PERIOD_US = 100_000

def make_tables(missing_rate=0.10, time_jitter_ms=4.0, seed=21):
    local = np.random.default_rng(seed)
    scene = pd.DataFrame([{'scene_token': 'scene-001', 'name': 'toy_scene', 'nbr_samples': 30}])
    sample = pd.DataFrame([
        {'sample_token': f'sample-{k:03d}', 'scene_token': 'scene-001', 'frame_id': k, 'timestamp_us': k * FRAME_PERIOD_US}
        for k in range(30)
    ])
    ego_pose = pd.DataFrame([
        {'ego_pose_token': f'pose-{k:03d}', 'timestamp_us': k * FRAME_PERIOD_US, 'x': 0.8 * k, 'y': 0.0, 'yaw': 0.01 * k}
        for k in range(30)
    ])
    calibrated_sensor = pd.DataFrame([
        {'calibrated_sensor_token': f'calib-{sensor}', 'sensor_name': sensor, 'translation': '[0,0,0]', 'rotation': '[1,0,0,0]'}
        for sensor in SENSORS
    ])
    rows = []
    for frame in range(30):
        for sensor in SENSORS:
            if local.random() < missing_rate:
                continue
            timestamp = frame * FRAME_PERIOD_US + int(local.normal(0, time_jitter_ms * 1000))
            rows.append({
                'sample_data_token': f'sd-{frame:03d}-{sensor}',
                'sample_token': f'sample-{frame:03d}',
                'sensor_name': sensor,
                'timestamp_us': timestamp,
                'ego_pose_token': f'pose-{frame:03d}',
                'calibrated_sensor_token': f'calib-{sensor}',
                'filename': f'samples/{sensor}/{frame:03d}.bin',
            })
    sample_data = pd.DataFrame(rows)
    return scene, sample, sample_data, ego_pose, calibrated_sensor

scene, sample, sample_data, ego_pose, calibrated_sensor = make_tables()
print('sample_data rows:', len(sample_data))
print(sample_data.head(3).to_string(index=False))



In [ ]:
def audit_tables(scene, sample, sample_data, ego_pose, calibrated_sensor, tolerance_ms=20.0):
    expected = {(frame, sensor) for frame in sample['frame_id'] for sensor in SENSORS}
    observed = {(int(row.sample_token.split('-')[1]), row.sensor_name) for row in sample_data.itertuples()}
    missing = sorted(expected - observed)
    valid_pose = set(ego_pose['ego_pose_token'])
    valid_calib = set(calibrated_sensor['calibrated_sensor_token'])
    broken_pose = sample_data[~sample_data['ego_pose_token'].isin(valid_pose)]
    broken_calib = sample_data[~sample_data['calibrated_sensor_token'].isin(valid_calib)]
    offsets = sample_data['timestamp_us'] - sample_data['sample_token'].str.split('-').str[1].astype(int) * FRAME_PERIOD_US
    bad_time = sample_data[np.abs(offsets) > tolerance_ms * 1000].copy()
    bad_time['offset_ms'] = offsets[np.abs(offsets) > tolerance_ms * 1000] / 1000.0
    return {
        'missing': missing,
        'broken_pose': broken_pose,
        'broken_calib': broken_calib,
        'bad_time': bad_time,
        'offsets_ms': offsets / 1000.0,
    }

audit = audit_tables(scene, sample, sample_data, ego_pose, calibrated_sensor)
print('missing pairs:', len(audit['missing']))
print('bad timestamp rows:', len(audit['bad_time']))



In [ ]:
def get_bundle(frame_id, tables, tolerance_ms=20.0):
    scene, sample, sample_data, ego_pose, calibrated_sensor = tables
    target_us = frame_id * FRAME_PERIOD_US
    rows = sample_data[sample_data['sample_token'] == f'sample-{frame_id:03d}'].copy()
    rows['offset_ms'] = (rows['timestamp_us'] - target_us) / 1000.0
    rows['within_tolerance'] = np.abs(rows['offset_ms']) <= tolerance_ms
    rows = rows.merge(ego_pose, on='ego_pose_token', how='left', suffixes=('', '_pose'))
    rows = rows.merge(calibrated_sensor, on='calibrated_sensor_token', how='left')
    present = set(rows['sensor_name']) if 'sensor_name' in rows else set()
    bundle = []
    for sensor in SENSORS:
        if sensor in present:
            row = rows[rows['sensor_name'] == sensor].iloc[0].to_dict()
            bundle.append(row)
        else:
            bundle.append({
                'sensor_name': sensor,
                'filename': None,
                'offset_ms': np.nan,
                'within_tolerance': False,
            })
    return pd.DataFrame(bundle)

bundle = get_bundle(0, (scene, sample, sample_data, ego_pose, calibrated_sensor))
print(bundle[['sensor_name', 'filename', 'offset_ms', 'within_tolerance']].to_string(index=False))




### 练习：把数据错误变成显式状态

- 调节 missing_rate 和 time_jitter_ms，统计每个 sensor 的可用率和时间偏移 p95。
- 把一条 calibration token 改成不存在的值，验证 audit 是否能拒绝它。
- 为 bundle 增加 quality 字段：missing、stale、valid、calibration_error。
- 对真实 nuScenes，找出 table token 关系和本 toy schema 的差异，不要把 toy 字段直接当成官方 API。


In [ ]:
def show_data_quality(missing_rate=0.10, time_jitter_ms=4.0, tolerance_ms=20.0, frame_id=10):
    tables = make_tables(missing_rate=missing_rate, time_jitter_ms=time_jitter_ms)
    scene, sample, sample_data, ego_pose, calibrated_sensor = tables
    result = audit_tables(*tables, tolerance_ms=tolerance_ms)
    availability = sample_data.assign(available=1).pivot_table(
        index='sensor_name', columns='sample_token', values='available', aggfunc='max', fill_value=0
    )
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].imshow(availability.reindex(SENSORS).values, aspect='auto', cmap='Greens')
    ax[0].set_yticks(range(len(SENSORS)), SENSORS)
    ax[0].set_xlabel('frame')
    ax[0].set_title(f'availability; missing={len(result["missing"])}')
    ax[1].hist(result['offsets_ms'], bins=20)
    ax[1].axvline(tolerance_ms, color='tab:red', linestyle='--')
    ax[1].axvline(-tolerance_ms, color='tab:red', linestyle='--')
    ax[1].set_title(f'timestamp offsets; bad={len(result["bad_time"])}')
    ax[1].set_xlabel('offset / ms')
    plt.tight_layout()
    plt.show()
    print(get_bundle(frame_id, tables, tolerance_ms)[['sensor_name', 'offset_ms', 'within_tolerance']].to_string(index=False))

interact(
    show_data_quality,
    missing_rate=FloatSlider(min=0.0, max=0.5, step=0.05, value=0.10, description='missing'),
    time_jitter_ms=FloatSlider(min=0.0, max=80.0, step=5.0, value=4.0, description='jitter ms'),
    tolerance_ms=FloatSlider(min=5.0, max=100.0, step=5.0, value=20.0, description='tolerance'),
    frame_id=IntSlider(min=0, max=29, step=1, value=0, description='frame'),
);



## 完成标准

- 输出一个包含 sensor filename、timestamp offset、pose 和 calibration 的 bundle。
- 报告 missing、stale timestamp、broken calibration 的数量。
- 解释为什么“能读到文件”不等于“输入对齐正确”。
- 下一步把 make_tables 替换为真实 nuScenes / Waymo adapter，并保留同样的 audit 接口。
